In [2]:
import pandas as pd
import re

In [3]:
df = pd.read_csv("../data/email_evaluation_dataset_Abhay-Mulani.csv")

df.head()

,email_text,expected_action,expected_tone
0,Reminder: Project review meeting scheduled tom...,notify,neutral
1,Please find attached the invoice for your Augu...,respond,urgent
2,Thank you for registering for our webinar. No ...,ignore,polite
3,Can you please share the updated design docume...,respond,urgent
4,This is to inform you that the office will rem...,notify,neutral


In [4]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_email"] = df["email_text"].apply(clean_text)


In [ ]:
IMPORTANT_KEYWORDS = ["urgent", "submit", "deadline"]
THANK_YOU_KEYWORDS = ["thank you", "thanks", "thankyou"]

def email_assistant(email):
    # urgent emails
    for word in IMPORTANT_KEYWORDS:
        if word in email:
            return "notify", "urgent"
    
    # polite / thank-you emails
    for word in THANK_YOU_KEYWORDS:
        if word in email:
            return "ignore", "polite"
    
    # default normal emails
    return "respond", "neutral"


In [6]:
df[["email_text", "predicted_action"]].head(10)

,email_text,predicted_action
0,Reminder: Project review meeting scheduled tom...,"(respond, neutral)"
1,Please find attached the invoice for your Augu...,"(respond, neutral)"
2,Thank you for registering for our webinar. No ...,"(ignore, polite)"
3,Can you please share the updated design docume...,"(respond, neutral)"
4,This is to inform you that the office will rem...,"(respond, neutral)"
5,Your OTP for account login is 482913. Do not s...,"(respond, neutral)"
6,We noticed unusual activity on your account. P...,"(respond, neutral)"
7,Monthly newsletter: Top tech trends you should...,"(respond, neutral)"
8,Your interview has been scheduled for Monday a...,"(respond, neutral)"
9,Reminder: Assignment submission deadline is to...,"(notify, urgent)"


In [7]:
df[["predicted_action", "predicted_tone"]] = (
    df["clean_email"]
    .apply(email_assistant)
    .apply(pd.Series)
)


In [8]:
df["action_correct"] = df["predicted_action"] == df["expected_action"]
df["tone_correct"] = df["predicted_tone"] == df["expected_tone"]


In [9]:
action_accuracy = df["action_correct"].mean() * 100
tone_accuracy = df["tone_correct"].mean() * 100

action_accuracy, tone_accuracy


(41.333333333333336, 65.33333333333333)

In [10]:
errors = df[df["action_correct"] == False]

errors[[
    "email_text",
    "expected_action",
    "predicted_action"
]].head(10)


,email_text,expected_action,predicted_action
0,Reminder: Project review meeting scheduled tom...,notify,respond
4,This is to inform you that the office will rem...,notify,respond
5,Your OTP for account login is 482913. Do not s...,notify,respond
7,Monthly newsletter: Top tech trends you should...,ignore,respond
12,Team lunch planned this Friday at 1 PM.,notify,respond
13,Your password was changed successfully.,notify,respond
14,Limited-time offer! Get 50% off on premium plans.,ignore,respond
16,This email is to acknowledge receipt of your a...,ignore,respond
17,Server maintenance scheduled tonight from 12 A...,notify,respond
18,Urgent: Action required to avoid service disru...,respond,notify


In [12]:
df.to_csv(
    "../data/milestone2_output_AbhayMulani.csv",
    index=False
)


### Reflection

**Which type of emails were hardest to classify?**  
Emails that required action but did not contain explicit keywords like "urgent" or "submit" were the hardest to classify.

**Why did your rules fail in some cases?**  
The rule-based system relies only on keyword matching and fails to understand context, intent, and implicit urgency, leading to low action accuracy (41.33%).

**How could an LLM improve this process?**  
An LLM can understand semantic meaning and intent beyond keywords, allowing it to infer urgency and required actions more accurately, improving overall classification.
